# Sample Employee Data Generator
Generates synthetic employee records and loads them to `main.dot_gold.dot_employee`.

In [0]:
%pip install faker
dbutils.library.restartPython()

In [0]:
from faker import Faker
from datetime import date, timedelta
import random
from pyspark.sql.types import StructType, StructField, StringType, DateType, DoubleType

fake = Faker()
Faker.seed(42)
random.seed(42)

NUM_EMPLOYEES = 500

DEPARTMENTS = [
    "Engineering", "Operations", "Finance", "Human Resources",
    "Planning", "Safety", "Maintenance", "IT", "Legal", "Executive"
]

JOB_TITLES = {
    "Engineering":      ["Civil Engineer", "Structural Engineer", "Traffic Engineer", "Senior Engineer", "Engineering Manager"],
    "Operations":       ["Operations Analyst", "Field Inspector", "Operations Manager", "Dispatch Coordinator", "Crew Supervisor"],
    "Finance":          ["Budget Analyst", "Accountant", "Financial Manager", "Procurement Specialist", "Auditor"],
    "Human Resources":  ["HR Specialist", "Recruiter", "HR Manager", "Benefits Coordinator", "Training Specialist"],
    "Planning":         ["Transportation Planner", "GIS Analyst", "Project Manager", "Urban Planner", "Planning Director"],
    "Safety":           ["Safety Inspector", "Safety Manager", "Compliance Officer", "Risk Analyst", "Safety Director"],
    "Maintenance":      ["Maintenance Technician", "Equipment Operator", "Maintenance Supervisor", "Fleet Manager", "Road Crew Lead"],
    "IT":               ["Systems Administrator", "Data Analyst", "Software Developer", "IT Manager", "Database Administrator"],
    "Legal":            ["Legal Counsel", "Paralegal", "Contracts Specialist", "Compliance Analyst", "General Counsel"],
    "Executive":        ["Division Director", "Deputy Director", "Chief Engineer", "Assistant Secretary", "Secretary of Transportation"],
}

SALARY_RANGES = {
    "Engineering":      (65000, 140000),
    "Operations":       (45000, 105000),
    "Finance":          (55000, 120000),
    "Human Resources":  (50000, 110000),
    "Planning":         (60000, 130000),
    "Safety":           (55000, 115000),
    "Maintenance":      (38000, 85000),
    "IT":               (60000, 135000),
    "Legal":            (70000, 160000),
    "Executive":        (95000, 200000),
}

rows = []
for i in range(1, NUM_EMPLOYEES + 1):
    emp_id = f"EMP-{i:06d}"
    sex = random.choice(["M", "F"])
    name = fake.name_male() if sex == "M" else fake.name_female()
    birth_date = fake.date_between(start_date=date(1960, 1, 1), end_date=date(2002, 12, 31))
    earliest_hire = birth_date + timedelta(days=18 * 365)
    hire_start = max(earliest_hire, date(2010, 1, 1))
    hire_date = fake.date_between(start_date=hire_start, end_date=date(2025, 12, 31))
    ssn = fake.ssn()
    department = random.choice(DEPARTMENTS)
    job_title = random.choice(JOB_TITLES[department])
    sal_min, sal_max = SALARY_RANGES[department]
    years_tenure = (date(2026, 1, 1) - hire_date).days / 365.25
    salary = round(random.uniform(sal_min, sal_max) * (1 + years_tenure * 0.02), 2)
    first_last = name.lower().replace("dr. ", "").replace("mr. ", "").replace("mrs. ", "").replace("ms. ", "").replace(" jr.", "").replace(" sr.", "").replace(" ", ".")
    email = f"{first_last}@dot.gov"
    phone = fake.phone_number()
    rows.append((emp_id, name, sex, birth_date, hire_date, ssn, department, job_title, salary, email, phone))

schema = StructType([
    StructField("employee_id", StringType(), False),
    StructField("name", StringType(), False),
    StructField("sex", StringType(), False),
    StructField("birth_date", DateType(), False),
    StructField("hire_date", DateType(), False),
    StructField("ssn", StringType(), False),
    StructField("department", StringType(), False),
    StructField("job_title", StringType(), False),
    StructField("salary", DoubleType(), False),
    StructField("email", StringType(), False),
    StructField("phone_number", StringType(), False),
])

df_employees = spark.createDataFrame(rows, schema=schema)
print(f"Generated {df_employees.count()} employee records with {len(df_employees.columns)} columns")
df_employees.printSchema()

In [0]:
display(df_employees.limit(20))
print(f"\nTotal rows: {df_employees.count()}")
df_employees.printSchema()

In [0]:
df_employees.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("main.dot_gold.dot_employee")
print("✅ Loaded to main.dot_gold.dot_employee")

In [0]:
df_verify = spark.table("main.dot_gold.dot_employee")
print(f"Row count in main.dot_gold.dot_employee: {df_verify.count()}")
display(df_verify.limit(10))